# Band 多batch 确定性方案 —— 验证与效率分析

## 三模式自适应（预处理：clamp + 行列裁剪）
先对 (m,n,p,q) 预处理：`p=min(p,m), q=min(q,n)`（clamp），
`m=min(m,n+p-1), n=min(n,m+q-1)`（裁剪空行空列）。然后分流：

- **Causal**（`causal_direction` 返回非None，方阵m=n）→ Mode03
  - **lower 左下三角** (x≥y)：正向causal，query看过去key。`pe>qe`
  - **upper 右上三角** (x≤y)：反向causal，query看未来key。`pe<qe`
- **Dense**（非三角 且 `p+q>m`）→ 列优先
- **Band**（其余：右上/m≠n三角/一般带）→ batch内集中+配对 ⭐本文档重点

## Band确定性约束（batch内严格）
1. 同batch同**行**块 → 不同轮次
2. 同batch同**列**块 → 不同轮次
3. **整列分核**：同batch同列由一个核处理
4. 跨batch之间：无约束

## 核心机制
- **对角线公式** `x = y + r2 - q`：同轮次不同列的核，因y不同→x不同→行不冲突
- **slot = p+q-1**（非Λ）：保证 p+q>m 时短列（右上三角）全部有效行可覆盖

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib
from collections import defaultdict
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

import sys
sys.path.insert(0, '/home/f00965199/FAG/der/code')
from band_hybrid import (
    calc_pos_hybrid, seg_params, col_range,
    is_causal, causal_direction, should_use_dense, find_perfect_pairs
)

## 1. 确定性验证函数

In [ ]:
def verify_band(k, m, n, p, q, b):
    """验证Band确定性 + 效率分析"""
    P = seg_params(m, n, p, q)
    n_seg, Lam = P['n_seg'], P['Lam']
    max_r = (m * n_seg * b // k + 10) * Lam

    mapping = []
    for j in range(1, k+1):
        for r in range(1, max_r+1):
            res = calc_pos_hybrid(k, m, n, p, q, b, j, r)
            if res:
                w, (x, y) = res
                if 1<=w<=b and 1<=x<=m and 1<=y<=n_seg:
                    mapping.append((w, j, r, x, y))

    errors = []
    # 同batch同行轮次不重复
    row_r = defaultdict(lambda: defaultdict(list))
    for w,j,r,x,y in mapping: row_r[w][x].append(r)
    for w,rows in row_r.items():
        for x,rs in rows.items():
            if len(rs)!=len(set(rs)): errors.append(f'b{w}行{x}轮次重复')
    # 同batch同列轮次不重复
    col_r = defaultdict(lambda: defaultdict(list))
    for w,j,r,x,y in mapping: col_r[w][y].append(r)
    for w,cols in col_r.items():
        for y,rs in cols.items():
            if len(rs)!=len(set(rs)): errors.append(f'b{w}列{y}轮次重复')
    # 整列分核
    col_c = defaultdict(lambda: defaultdict(set))
    for w,j,r,x,y in mapping: col_c[w][y].add(j)
    for w,cols in col_c.items():
        for y,cs in cols.items():
            if len(cs)>1: errors.append(f'b{w}列{y}多核{cs}')
    # 覆盖完整
    expected = {(w,x,y) for w in range(1,b+1) for y in range(1,n_seg+1)
                for x in range(col_range(y,P)[0], col_range(y,P)[1]+1)}
    got = {(w,x,y) for w,j,r,x,y in mapping}
    if expected!=got: errors.append(f'覆盖缺{len(expected-got)}')

    total_blocks = len(expected)
    actual_r = max((r for _,_,r,_,_ in mapping), default=0)
    theo_r = (total_blocks + k - 1) // k
    eff = theo_r/actual_r if actual_r>0 else 0
    return len(errors)==0, {'blocks':total_blocks,'theo':theo_r,
            'actual':actual_r,'eff':eff,'errors':errors}

## 2. Causal 方向判断（左下/右上）

`causal_direction(m,n,p,q)` 判断三角方向：
- `'lower'` 左下(x≥y) 正向causal；`'upper'` 右上(x≤y) 反向causal；`'None'` 非三角

In [ ]:
def show_dir(m,n,p,q,desc):
    cd = causal_direction(m,n,p,q)
    P = seg_params(m,n,p,q)
    # 实际形状
    blk=set()
    for y in range(1,P['n_seg']+1):
        xs,xe=col_range(y,P)
        for x in range(xs,xe+1):
            if 1<=x<=P['m'] and 1<=y<=P['n']: blk.add((x,y))
    if not blk: shape='空'
    elif all(x>=y for x,y in blk): shape='左下(x≥y)'
    elif all(x<=y for x,y in blk): shape='右上(x≤y)'
    else: shape='带状'
    print(f'  {desc:<16} p={p},q={q}: 方向={str(cd):<6} 实际形状={shape}')

print('causal_direction 判断:')
show_dir(8,8,8,1,'标准左下(正)')
show_dir(8,8,1,8,'标准右上(反)')
show_dir(8,8,7,2,'近左下')
show_dir(8,8,2,7,'近右上')
show_dir(8,8,5,5,'对称(非三角)')
show_dir(12,15,4,5,'一般带(非三角)')

In [ ]:
# 可视化两种三角方向
fig,axes=plt.subplots(1,4,figsize=(13,3.3))
for ax,(m,n,p,q,t) in zip(axes,[
    (8,8,8,1,'左下lower(正causal)'),(8,8,1,8,'右上upper(反causal)'),
    (8,8,7,2,'近左下'),(8,8,2,7,'近右上')]):
    P=seg_params(m,n,p,q); mask=np.zeros((P['m'],P['n']))
    for y in range(1,P['n_seg']+1):
        xs,xe=col_range(y,P)
        for x in range(xs,xe+1):
            if 1<=x<=P['m'] and 1<=y<=P['n']: mask[x-1,y-1]=1
    ax.imshow(mask,origin='upper',cmap='Greens',vmin=0,vmax=1.3,aspect='equal')
    ax.set_xticks(np.arange(P['n']+1)-0.5,minor=True)
    ax.set_yticks(np.arange(P['m']+1)-0.5,minor=True)
    ax.grid(which='minor',color='gray',linewidth=0.5)
    ax.tick_params(which='minor',length=0); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{t}\ndir={causal_direction(m,n,p,q)}',fontsize=9)
    ax.set_xlabel('y(key)',fontsize=8); ax.set_ylabel('x(query)',fontsize=8)
plt.suptitle('Causal两方向: 左下(x≥y)正向 / 右上(x≤y)反向',fontsize=11,y=1.02)
plt.tight_layout(); plt.show()

## 3. 效率对比表

**理论最优轮次** = ⌈总有效块数 / k⌉（完美负载均衡下界）
**实际最大轮次** = 本方案的最大轮次
**效率** = 理论最优 / 实际

In [ ]:
band_cases = [    # (k, m, n, p, q, b, 描述)    (6, 8, 6, 1, 3, 4, '窄带'),    (5, 12, 15, 4, 5, 6, '标准梯形'),    (6, 8, 10, 3, 4, 4, '中梯形'),    (8, 5, 5, 3, 3, 4, '对称三角'),    (6, 6, 6, 2, 4, 4, '左偏三角'),    (8, 6, 8, 3, 3, 4, '小梯形'),    (13, 17, 28, 9, 8, 8, '大梯形'),    (24, 14, 20, 2, 12, 4, 'q大(右上→Band)'),    (25, 18, 7, 4, 9, 4, 'q>n右上→Band'),    (6, 24, 10, 9, 14, 10, 'q>n大(clamp)'),    (10, 20, 20, 8, 8, 6, '大方阵band'),    (16, 9, 11, 7, 1, 8, 'p大q小(m≠n→Band)'),]def dispatch_mode(m, n, p, q):    """与 calc_pos_hybrid 分流精确对齐"""    cd = causal_direction(m, n, p, q)    if cd == 'lower' and m == n:        return 'Causal'          # 左下+方阵 → Mode03    if cd is None and should_use_dense(seg_params(m, n, p, q)):        return 'Dense'           # 非三角+近满 → Dense    return 'Band'                # 右上/m≠n三角/一般带 → Bandprint(f"{'描述':<18}{'k':<4}{'m':<4}{'n':<4}{'p':<4}{'q':<4}{'b':<4}"      f"{'总块数':<8}{'理论轮次':<10}{'实际轮次':<10}{'效率':<8}{'模式':<8}{'确定性':<6}")print('-'*110)all_ok = Trueband_count = 0for k,m,n,p,q,b,desc in band_cases:    mode = dispatch_mode(m, n, p, q)    if mode == 'Band':        band_count += 1        ok, d = verify_band(k,m,n,p,q,b)        all_ok = all_ok and ok        status = '✅' if ok else '❌'        print(f'{desc:<18}{k:<4}{m:<4}{n:<4}{p:<4}{q:<4}{b:<4}'              f"{d['blocks']:<8}{d['theo']:<10}{d['actual']:<10}"              f"{d['eff']*100:>6.1f}% {mode:<8}{status:<6}")    else:        print(f'{desc:<18}{k:<4}{m:<4}{n:<4}{p:<4}{q:<4}{b:<4}'              f"{'--':<8}{'--':<10}{'--':<10}{'--':<8}{mode:<8}⊘跳过")print('-'*110)print(f"Band分支: {band_count}个全部通过 ✅" if all_ok else f"Band分支: 有失败")print(f"非Band(Causal/Dense): {len(band_cases)-band_count}个跳过（需单独实现）")

## 4. 随机压力测试

In [ ]:
import random
random.seed(2024)
passed=total=0; effs=[]; failed=[]
tries=0
while total<100 and tries<3000:
    tries+=1
    m=random.randint(4,16); n=random.randint(4,20)
    p=random.randint(1,m+2); q=random.randint(1,n+2)  # 含clamp边界
    b=random.randint(2,8); k=random.randint(2,20)
    if is_causal(m,n,p,q): continue
    P=seg_params(m,n,p,q)
    if should_use_dense(P): continue
    if P['n_seg']<1: continue
    total+=1
    ok,d=verify_band(k,m,n,p,q,b)
    if ok: passed+=1; effs.append(d['eff'])
    else: failed.append((k,m,n,p,q,b))

print(f'随机测试: {passed}/{total} ({passed/total*100:.1f}%)')
if effs: print(f'效率: 平均{sum(effs)/len(effs)*100:.1f}% 最低{min(effs)*100:.1f}% 最高{max(effs)*100:.1f}%')
if failed: print(f'失败: {failed[:5]}')

## 5. 可视化

In [ ]:
def visualize(k, m, n, p, q, b, title=''):
    P = seg_params(m, n, p, q)
    n_seg = P['n_seg']
    max_r = (m * n_seg * b // k + 10) * P['Lam']
    mapping = []
    for j in range(1, k+1):
        for r in range(1, max_r+1):
            res = calc_pos_hybrid(k, m, n, p, q, b, j, r)
            if res:
                w,(x,y)=res
                if 1<=w<=b and 1<=x<=m and 1<=y<=n_seg:
                    mapping.append((w,j,r,x,y))

    ncols=min(b,4); nrows=math.ceil(b/ncols)
    fig,axes=plt.subplots(nrows,ncols,figsize=(3*ncols,3*nrows*m/n_seg),squeeze=False)
    cmap=plt.get_cmap('tab20',k)
    for w in range(1,b+1):
        ax=axes[(w-1)//ncols][(w-1)%ncols]
        cm=np.zeros((m,n_seg),int); rm=np.zeros((m,n_seg),int)
        for ww,j,r,x,y in mapping:
            if ww==w: cm[x-1,y-1]=j; rm[x-1,y-1]=r
        masked=np.ma.array(cm,mask=(cm==0))
        ax.imshow(masked,origin='upper',cmap=cmap,vmin=1,vmax=k,aspect='auto')
        ax.set_xticks(np.arange(n_seg+1)-0.5,minor=True)
        ax.set_yticks(np.arange(m+1)-0.5,minor=True)
        ax.grid(which='minor',color='black',linewidth=0.5)
        ax.tick_params(which='minor',length=0)
        for i in range(m):
            for c in range(n_seg):
                if rm[i,c]>0: ax.text(c,i,str(rm[i,c]),ha='center',va='center',fontsize=6,color='white',weight='bold')
        rs=[rm[i,c] for i in range(m) for c in range(n_seg) if rm[i,c]>0]
        ax.set_title(f'B{w} r∈[{min(rs)},{max(rs)}]' if rs else f'B{w}',fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
    for idx in range(b,nrows*ncols): axes[idx//ncols][idx%ncols].axis('off')
    pairs=find_perfect_pairs(P)
    fig.suptitle(f'{title}  k={k} m={m} n={n} p={p} q={q} b={b}\n'
                 f'L1={P["L1"]} L2={P["L2"]} L3={P["L3"]} 配对={len(pairs)}',fontsize=9)
    fig.tight_layout(rect=[0,0,1,0.95]); plt.show()

### 标准梯形

In [ ]:
visualize(5, 12, 15, 4, 5, 4, '标准梯形')

### 三角形

In [ ]:
visualize(8, 5, 5, 3, 3, 4, '对称三角')

### q>n (clamp边界)

In [ ]:
visualize(6, 8, 6, 4, 9, 4, 'q>n-clamp')